# Preparar el entorno de trabajo

In [1]:
!pip install openai pydantic

import os
import json
from datetime import date
from typing import List
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 40.5 MB/s  0:00:00

  Attempting uninstall: idna

    Found existing installation: idna 3.10

    Uninstalling idna-3.10:

      Successfully uninstalled idna-3.10

   -------------------- ------------------- 3/6 [httpcore2]
   -------------------------- ------------- 4/6 [httpx2]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   --------------------------------- ------ 5/6 [openai]
   ---------------------

# Definir los schemas

In [2]:
# Definición de schemas utilizando Pydantic para garantizar la estructura del JSON

class GenerationMetadata(BaseModel):
    """Clase para metadatos adicionales de la generación."""
    generation_date: str = Field(description="Fecha de generación de los datos en formato YYYY-MM-DD")
    target_audience: str = Field(description="Público objetivo principal al que va dirigido el SEO")
    content_language: str = Field(description="Idioma del contenido generado")

class SEOProductDescription(BaseModel):
    """Clase principal para describir los productos y su optimización SEO."""
    name: str = Field(description="Nombre comercial y atractivo del producto ficticio")
    description: str = Field(description="Descripción detallada del producto orientada a la venta (mínimo 3 oraciones)")
    seo_meta_description: str = Field(description="Meta descripción SEO (máximo 160 caracteres)")
    keywords: List[str] = Field(description="Lista de 5 a 8 palabras clave de cola larga (long-tail keywords)")
    seo_score: int = Field(description="Puntuación estimada de optimización SEO del 1 al 100")

class StructuredProductOutput(BaseModel):
    """Clase raíz que agrupa metadatos y la información del producto."""
    metadata: GenerationMetadata
    product: SEOProductDescription

# Realizar la llamada a la API de OpenAI y Configuración
 

In [3]:
# Configuración de la clave API (Ficticia según los requisitos)
os.environ["OPENAI_API_KEY"] = "sk-YOUR_FAKE_API_KEY"

# Inicialización del cliente de OpenAI
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def generate_seo_content(product_concept: str) -> StructuredProductOutput:
    """
    Realiza una llamada a la API de OpenAI utilizando el formato de salidas estructuradas (Structured Outputs).
    """
    system_prompt = (
        "Eres un experto en marketing digital y consultor SEO para e-commerce. "
        "Tu objetivo es recibir el concepto de un producto y devolver datos estructurados "
        "con descripciones altamente optimizadas para motores de búsqueda, asegurando "
        "que el contenido sea persuasivo y cumpla estrictamente con el esquema JSON proporcionado."
    )
    
    try:
        # Se utiliza el método beta.chat.completions.parse soportado en las versiones recientes de OpenAI
        # para forzar la salida estructurada validada con Pydantic.
        response = client.beta.chat.completions.parse(
            model="gpt-4o-mini", # Modelo eficiente y compatible con structured outputs
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Genera el contenido SEO y metadatos para el siguiente concepto: {product_concept}. La fecha de hoy es {date.today().isoformat()}."}
            ],
            response_format=StructuredProductOutput,
            temperature=0.7
        )
        
        # Retorna directamente el objeto validado por Pydantic
        return response.choices[0].message.parsed
        
    except Exception as e:
        # Captura errores a nivel de API (ej. clave inválida, problemas de red)
        print(f"Error en la comunicación con la API de OpenAI: {str(e)}")
        return None

# Validar las respuestas y Caso de uso práctico

In [4]:
def validate_and_display_use_case(product_concept: str):
    """
    Ejecuta el caso de uso práctico, capturando explícitamente errores de validación de Pydantic.
    """
    print(f"--- Generando contenido para: '{product_concept}' ---\n")
    
    try:
        # Aquí se ejecutaría la llamada. 
        # Nota: Como la API Key es ficticia, esto lanzará la excepción de la API definida arriba.
        # Para probar la validación localmente sin gastar tokens, podemos simular la respuesta validándola manualmente:
        
        resultado_estructurado = generate_seo_content(product_concept)
        
        if resultado_estructurado:
            # Serializamos a JSON con indentación para demostrar que cumple el esquema
            json_output = resultado_estructurado.model_dump_json(indent=2)
            print("✅ JSON Validado Correctamente:")
            print(json_output)
            
    except ValidationError as e:
        print("❌ Error de validación: El JSON devuelto por la API no cumple con el esquema Pydantic.")
        print(e.json())

# Probar el proyecto (Múltiples iteraciones)

In [5]:
productos_prueba = [
    "Reloj inteligente híbrido con carga solar infinita y monitorización de apnea del sueño",
    "Servicio premium de limpieza de hogar con productos 100% probióticos y ecológicos",
    "Teclado mecánico dividido ergonómico para programadores con switches silenciosos"
]

# Ejecutar las pruebas
for producto in productos_prueba:
    validate_and_display_use_case(producto)
    print("\n" + "="*60 + "\n")

--- Generando contenido para: 'Reloj inteligente híbrido con carga solar infinita y monitorización de apnea del sueño' ---

Error en la comunicación con la API de OpenAI: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-YOUR_********_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


--- Generando contenido para: 'Servicio premium de limpieza de hogar con productos 100% probióticos y ecológicos' ---

Error en la comunicación con la API de OpenAI: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-YOUR_********_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


--- Generando contenido para: 'Teclado mecánico dividido ergonómico para programadores con switches silenciosos' ---

Error en la comunicació

# Análisis del caso de uso práctico en E-commerce

## Análisis: Implementación en un entorno de E-commerce
La integración de este script en la arquitectura de un comercio electrónico real automatiza uno de los cuellos de botella más grandes del catálogo: el alta de productos y el posicionamiento orgánico.

1. Eficiencia a escala: Al recibir un flujo de datos (ej. un CSV con nombres básicos de proveedor), este pipeline puede enriquecer cientos de productos diarios con descripciones persuasivas y metadatos SEO sin intervención manual.

2. Consistencia de datos: Gracias a la validación estricta con Pydantic y el soporte de Structured Outputs de OpenAI, garantizamos que el CMS (como Shopify o Magento) nunca reciba un JSON truncado o claves inexistentes que romperían la base de datos o el frontend.

3. Optimización algorítmica: La generación forzada de long-tail keywords y la limitación de la longitud de la seo_meta_description (160 caracteres) aseguran que el contenido generado cumple automáticamente con las mejores prácticas de indexación de Google.